# CardioSense Phase 1 — Pipeline 1: Clinical Risk Prediction

**UCI Heart Disease (Cleveland) → Logistic Regression → XGBoost → Calibration → SHAP**

Run `00_colab_setup.ipynb` first. This notebook runs top to bottom with no edits.

### Sections
1. Environment Setup · 2. Imports · 3. Configuration · 4. Dataset Verification
5. Exploratory Data Analysis · 6. Preprocessing · 7. Dataset Splitting · 8. Baseline
9. Model Definition · 10. Training · 11. Validation · 12. Evaluation
13. Explainability · 14. Calibration · 15. Error Analysis · 16. Save Model
17. Save Results · 18. Example Inference

### The three experiments this produces
| ID | Experiment | Question it answers |
|---|---|---|
| C-A | Logistic Regression | What does an interpretable linear model achieve? |
| C-B | XGBoost | Do non-linear interactions help on 303 patients? |
| C-C | Calibrated model | Are the probabilities themselves trustworthy? |

CPU-only — no GPU needed. Total runtime ≈ 2 minutes.

## 1. Environment Setup

In [ ]:
# On Colab, this cell wires up the repo. Locally it is a no-op.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    REPO_DIR = Path("/content/CardioSense")
    if not REPO_DIR.exists():
        raise RuntimeError("Repo not found. Run 00_colab_setup.ipynb first.")
    os.chdir(REPO_DIR)

    os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"
    subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=False)
    subprocess.run(["pip", "install", "-q", "-e", "."], check=False)
else:
    # Running locally: walk up to the project root.
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

print("Working directory:", Path.cwd())
print("Data root        :", os.environ.get("CARDIOSENSE_DATA_ROOT", "<repo>/data"))

## 2. Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cardiosense.common.config import load_config
from cardiosense.common.env import print_environment
from cardiosense.common.experiment import ExperimentTracker, load_experiment_log
from cardiosense.common.io_utils import save_json, save_pickle
from cardiosense.common.paths import PATHS
from cardiosense.common.seeding import set_seed

from cardiosense.clinical.data import FEATURE_DESCRIPTIONS, load_raw_dataframe, prepare_dataset
from cardiosense.clinical.eda import run_eda
from cardiosense.clinical.preprocessing import (
    build_preprocessor, fit_preprocessor, split_data, transform_splits,
)
from cardiosense.clinical.models import tune_logistic_regression, tune_xgboost
from cardiosense.clinical import evaluate as ev
from cardiosense.clinical import calibrate as cal
from cardiosense.clinical.explain import run_shap_analysis
from cardiosense.clinical.predict import ClinicalPredictor

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
_ = print_environment()

## 3. Configuration

Everything is read from `configs/clinical_config.yaml`. Nothing is hard-coded in
this notebook, so a change made here for an experiment is a change to a file that
gets committed — which is what makes the run reproducible.

In [ ]:
cfg = load_config("clinical")

SEED = int(cfg.seed)
set_seed(SEED, strict=bool(cfg.get("strict_determinism", True)))

RESULTS = PATHS.root / cfg.output.results_dir
MODELS = PATHS.root / cfg.output.models_dir
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print(f"seed              : {SEED}")
print(f"dataset           : {cfg.dataset.name} (source={cfg.dataset.source})")
print(f"target            : {cfg.dataset.target_column} -> binary "
      f"{cfg.dataset.get('target_name')}")
print(f"split             : {1 - cfg.split.test_size - cfg.split.val_size:.0%} train / "
      f"{cfg.split.val_size:.0%} val / {cfg.split.test_size:.0%} test")
print(f"selection metric  : {cfg.selection.primary_metric}")
print(f"threshold policy  : {cfg.selection.threshold_strategy} "
      f"on {cfg.selection.threshold_tuning_data}")
print(f"results -> {RESULTS}")
print(f"models  -> {MODELS}")

## 4. Dataset Verification

`ucimlrepo` downloads dataset id 45 and caches it, so this cell is fast on reruns
and works offline afterwards.

In [ ]:
raw = load_raw_dataframe(cfg)

print(f"raw shape: {raw.shape}")
print(f"columns  : {list(raw.columns)}")
print(f"\nmissing values:\n{raw.isna().sum()[lambda s: s > 0].to_string() or '  none'}")
print(f"\nraw target `{cfg.dataset.target_column}` distribution "
      f"(0 = no disease, 1-4 = severity):")
print(raw[cfg.dataset.target_column].value_counts().sort_index().to_string())
raw.head()

In [ ]:
# Clean and define the target. Every transformation is recorded in the report.
X, y, data_report = prepare_dataset(raw, cfg)

print(f"target rule : {data_report['target_rule']}")
print(f"positive    : {data_report['positive_rate']:.1%} "
      f"({data_report['target_distribution']})")
print(f"duplicates  : {data_report['duplicates_dropped']} dropped")
print(f"final shape : {X.shape}")

pd.DataFrame({"feature": X.columns,
              "description": [FEATURE_DESCRIPTIONS.get(c, "") for c in X.columns]})

## 5. Exploratory Data Analysis

Note what is and is not plotted. A Pearson correlation matrix over all 13 columns
would treat `thal` (3 = normal, 6 = fixed defect, 7 = reversible defect) as if
7 were "more" than 3. It isn't a scale, it's a set of labels. So correlation is
computed over numeric features only, and association with the target uses the
appropriate statistic per feature type: point-biserial for numeric, Cramér's V
for categorical.

In [ ]:
eda = run_eda(X, y, cfg, RESULTS / "eda")

print(f"rows {eda['shape']['rows']}, features {eda['shape']['features']}, "
      f"positive rate {eda['positive_rate']:.1%}, "
      f"imbalance ratio {eda['imbalance_ratio']:.2f}")
if eda["highly_correlated_pairs"]:
    print("highly correlated numeric pairs:", eda["highly_correlated_pairs"])

pd.DataFrame(eda["target_association"]).head(13)

In [ ]:
# Display the saved EDA figures inline.
from IPython.display import Image, display

for name in ["class_distribution.png", "numeric_distributions.png",
             "correlation_numeric.png", "target_association.png",
             "categorical_vs_target.png"]:
    path = RESULTS / "eda" / name
    if path.exists():
        print(f"--- {name} ---")
        display(Image(filename=str(path)))

## 6. Dataset Splitting

Done **before** preprocessing, deliberately. Fitting a scaler or an imputer on the
full dataset and splitting afterwards lets the test patients influence the
training representation — the single most common way a tabular result gets
quietly inflated.

Each split has one job:
- **train** — fits the preprocessor, the models, the CV search
- **validation** — selects the model, fits the calibrator
- **test** — scored exactly once, at the end

In [ ]:
splits = split_data(X, y, cfg)

print(json.dumps(splits.summary["sizes"], indent=2))
print("positive rate per split:", splits.summary["positive_rate"])
print("index overlap (must be all zero):", splits.summary["index_overlap"])

y_train = splits.y_train.to_numpy()
y_val = splits.y_val.to_numpy()
y_test = splits.y_test.to_numpy()

## 7. Preprocessing

`ColumnTransformer`: median imputation + scaling for numeric, most-frequent
imputation + one-hot for categorical. Fitted on **train only**, then applied
unchanged everywhere else — including at inference, because this exact fitted
object is what gets pickled.

In [ ]:
preprocessor = fit_preprocessor(build_preprocessor(cfg), splits.X_train, splits.y_train)
X_train_t, X_val_t, X_test_t, feature_names = transform_splits(preprocessor, splits)

print(f"{splits.X_train.shape[1]} raw features -> {X_train_t.shape[1]} encoded features")
print(f"NaNs remaining: {int(np.isnan(X_train_t).sum())}")
print("\nencoded feature names:")
print(feature_names)

## 8. Baseline — Experiment C-A: Logistic Regression

The interpretable reference. Its coefficients are log-odds ratios, which is the
same language existing clinical risk scores are written in, so it is the model a
clinician can actually audit.

In [ ]:
experiments = {}

lr_search = tune_logistic_regression(X_train_t, y_train, cfg)
lr_model = lr_search["estimator"]
lr_val_prob = ev.predict_proba(lr_model, X_val_t)

experiments["logistic_regression"] = {
    **{k: v for k, v in lr_search.items() if k != "estimator"},
    "model": lr_model,
    "val": ev.evaluate_at_threshold(y_val, lr_val_prob, 0.5),
    "val_prob": lr_val_prob,
}

print(f"best C            : {lr_search['best_params']['C']}")
print(f"CV ROC-AUC        : {lr_search['cv_score']:.4f} +/- {lr_search['cv_score_std']:.4f}")
print(f"validation ROC-AUC: {experiments['logistic_regression']['val']['roc_auc']:.4f}")

In [ ]:
# The coefficients, as odds ratios. This is the interpretability the baseline buys.
odds = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lr_model.coef_[0],
    "odds_ratio": np.exp(lr_model.coef_[0]),
}).sort_values("coefficient", key=abs, ascending=False)
odds.head(12).round(3)

## 9. Model Definition & 10. Training — Experiment C-B: XGBoost

The search space is deliberately constrained to shallow trees (`max_depth` 2–4).
With ~212 training rows, depth-8 trees memorise the training set within a few
boosting rounds and the CV score then measures memorisation, not generalisation.

40 candidates × 5 folds = 200 fits. An exhaustive grid over the same space would
be several thousand — and would mostly be fitting cross-validation noise.

In [ ]:
xgb_search = tune_xgboost(X_train_t, y_train, cfg)
xgb_model = xgb_search["estimator"]
xgb_val_prob = ev.predict_proba(xgb_model, X_val_t)

experiments["xgboost"] = {
    **{k: v for k, v in xgb_search.items() if k != "estimator"},
    "model": xgb_model,
    "val": ev.evaluate_at_threshold(y_val, xgb_val_prob, 0.5),
    "val_prob": xgb_val_prob,
}

print(f"best params : {json.dumps(xgb_search['best_params'], indent=2)}")
print(f"CV ROC-AUC  : {xgb_search['cv_score']:.4f} +/- {xgb_search['cv_score_std']:.4f}")
print(f"train ROC-AUC (same folds): {xgb_search['cv_train_score']:.4f}  "
      f"<- a large gap vs CV means overfitting")
print(f"search time : {xgb_search['search_seconds']}s for {xgb_search['n_fits']} fits")

## 11. Validation — model comparison and selection

Selection is on **ROC-AUC over the validation split**, not accuracy. Accuracy
depends on an arbitrary 0.5 threshold and, on ~45 patients, moves in 2-point
jumps. ROC-AUC is threshold-free and uses the full ranking.

If the two models finish within `tie_tolerance` of each other, the **simpler** one
wins — at this sample size a 0.01 AUC gap is inside the noise, and the linear
model is easier to explain, easier to calibrate, and less likely to have latched
onto a spurious interaction.

In [ ]:
comparison = ev.build_comparison_table(
    experiments, metrics=list(cfg.selection.report_metrics)
)
comparison.to_csv(RESULTS / "model_comparison.csv", index=False)
comparison

In [ ]:
selected_name, decision = ev.select_model(experiments, cfg)
decision["best_params"] = experiments[selected_name].get("best_params", {})
save_json(decision, RESULTS / "model_selection.json")

selected_model = experiments[selected_name]["model"]
val_prob = experiments[selected_name]["val_prob"]

print(f"SELECTED: {selected_name}")
print(f"validation {decision['primary_metric']} scores: {decision['scores']}")
if decision["tie_broken"]:
    print(f"\ntie-break applied:\n{decision['tie_reason']}")

In [ ]:
# Operating threshold. Tuned on out-of-fold train predictions POOLED with
# validation (~250 points) rather than on 45 validation points alone — a
# threshold picked from 45 points is itself a very noisy estimate.
from sklearn.base import clone

oof_prob = None
if str(cfg.selection.threshold_tuning_data) == "cv_oof_plus_val":
    oof_prob = ev.out_of_fold_probabilities(
        clone(selected_model), X_train_t, y_train, cfg,
        folds=int(cfg.models.xgboost.cv_folds),
    )

threshold, threshold_info = ev.tune_threshold(
    y_val, val_prob, cfg, oof_y=y_train, oof_prob=oof_prob
)
print(json.dumps(threshold_info, indent=2))

## 14. Calibration — Experiment C-C

*(Run before the final test evaluation so the test split is touched exactly once.)*

**Why this section exists.** Phase 2 fuses three modalities by weighting each one
by how far it should be trusted. That only works if the number this pipeline emits
means what it says: if the model outputs 0.80 across a group of patients, about
80% of them should actually have disease. A model can rank patients perfectly
(AUC 0.95) and still be badly wrong about the *level*.

**Vocabulary, used consistently:**

| Term | What it is |
|---|---|
| `prediction probability` | Raw classifier output. A **score** — valid for ranking, AUC, thresholds. No frequency guarantee. |
| `calibrated confidence` | Post-calibration output. A **frequency claim** — among patients given p, about p are positive. This is what Phase 2 weights by. |

**How the method is chosen without cheating.** The final calibrator is fitted on
validation. So the *choice* between Platt and isotonic cannot also be made on
validation — isotonic, being non-parametric, would win by fitting it exactly. And
choosing on test is straight leakage. Instead: out-of-fold predictions on train
give ~200 honest (probability, outcome) pairs, and each candidate mapping is fitted
and scored on **disjoint** halves of those.

In [ ]:
method, method_report = cal.select_calibration_method(selected_model, X_train_t, y_train, cfg)
print(json.dumps(method_report, indent=2))

In [ ]:
calibrator = cal.fit_calibrator(selected_model, X_val_t, y_val, method=method)

test_prob_raw = ev.predict_proba(selected_model, X_test_t)
test_prob_cal = ev.predict_proba(calibrator, X_test_t)

calibration_metrics = cal.calibration_report(
    y_test, test_prob_raw, test_prob_cal, cfg, RESULTS, method=method, split_name="test"
)

print(f"\nBrier    {calibration_metrics['uncalibrated']['brier']:.4f} -> "
      f"{calibration_metrics['calibrated']['brier']:.4f}")
print(f"ECE      {calibration_metrics['uncalibrated']['ece']:.4f} -> "
      f"{calibration_metrics['calibrated']['ece']:.4f}")
print(f"log loss {calibration_metrics['uncalibrated']['log_loss']:.4f} -> "
      f"{calibration_metrics['calibrated']['log_loss']:.4f}")
print(f"\nmean predicted probability {calibration_metrics['calibrated']['mean_predicted_probability']:.3f} "
      f"vs observed positive rate {calibration_metrics['observed_positive_rate']:.3f}")

display(Image(filename=str(RESULTS / "calibration_curve.png")))

## 12. Evaluation — the test split, scored once

Everything above used only train and validation. This is the first and only time
the test split is scored.

Reported with **bootstrap 95% confidence intervals**, because on ~45 patients a
point estimate alone is misleading — the interval is typically ±0.10 AUC wide, and
saying so is the honest presentation.

In [ ]:
extra_curves = {
    name: (y_test, ev.predict_proba(blocks["model"], X_test_t))
    for name, blocks in experiments.items() if name != selected_name
}

test_metrics = ev.evaluate_final(
    selected_name, y_test, test_prob_raw, threshold, cfg, RESULTS, extra_curves=extra_curves
)
test_metrics["calibrated_at_tuned_threshold"] = ev.evaluate_at_threshold(
    y_test, test_prob_cal, threshold
)

tuned = test_metrics["at_tuned_threshold"]
default = test_metrics["at_default_threshold_0.5"]
ci = test_metrics["confidence_intervals_95pct"]

print(f"ROC-AUC  {ci['roc_auc']['point']:.3f}  95% CI [{ci['roc_auc']['lower']:.3f}, "
      f"{ci['roc_auc']['upper']:.3f}]")
print(f"PR-AUC   {ci['pr_auc']['point']:.3f}  95% CI [{ci['pr_auc']['lower']:.3f}, "
      f"{ci['pr_auc']['upper']:.3f}]\n")

pd.DataFrame({
    f"threshold={threshold:.2f}": {k: tuned[k] for k in
        ("accuracy", "precision", "recall", "specificity", "f1", "brier")},
    "threshold=0.50": {k: default[k] for k in
        ("accuracy", "precision", "recall", "specificity", "f1", "brier")},
}).round(4)

In [ ]:
for name in ["confusion_matrix.png", "roc_curve.png", "pr_curve.png"]:
    display(Image(filename=str(RESULTS / name)))

## 15. Error Analysis

Metrics tell you how often the model is wrong. This tells you *which* patients it
gets wrong, and how confidently.

The cases worth reading are the **confident errors**. A false negative at p = 0.05
is a patient the model was sure was healthy — clinically, the failure mode that
matters most.

In [ ]:
errors = ev.export_errors(
    splits.X_test, y_test, test_prob_raw, threshold, RESULTS / "errors", prefix="test"
)
save_json(errors, RESULTS / "errors" / "error_summary.json")

print(f"errors: {errors['n_errors']} of {errors['n_total']} ({errors['error_rate']:.1%})")
print(f"breakdown: {errors['counts']}")
print(f"mean |p - threshold| when correct: {errors['mean_confidence_when_correct']}")
print(f"mean |p - threshold| when wrong  : {errors['mean_confidence_when_wrong']}")

error_frame = pd.read_csv(RESULTS / "errors" / "test_errors.csv")
error_frame.head(10)

In [ ]:
# Where do the false negatives sit relative to correctly-classified patients?
y_pred_test = (test_prob_raw >= threshold).astype(int)
fig, ax = plt.subplots(figsize=(9, 4))
for label, mask, colour in [
    ("correct", y_pred_test == y_test, "tab:green"),
    ("false negative", (y_test == 1) & (y_pred_test == 0), "tab:red"),
    ("false positive", (y_test == 0) & (y_pred_test == 1), "tab:orange"),
]:
    if mask.sum():
        ax.scatter(test_prob_raw[mask], np.random.default_rng(0).normal(0, 0.04, mask.sum()),
                   label=f"{label} (n={int(mask.sum())})", alpha=0.75, s=45, color=colour)
ax.axvline(threshold, color="k", ls="--", label=f"threshold = {threshold:.2f}")
ax.set_xlabel("predicted probability of disease")
ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_title("Test predictions by outcome — errors clustered near the threshold are "
             "ambiguous cases;\nerrors far from it are the ones to investigate")
plt.show()

## 13. Explainability — SHAP

A SHAP value is a feature's contribution to the gap between *this* prediction and
the model's average prediction, averaged fairly over all orderings.

**What it does not tell you.** It explains the model, not the disease. `thalach`
and `age` are correlated; the model may load its reliance onto either one, and
SHAP will honestly report whichever it chose. A low SHAP importance therefore does
**not** establish clinical irrelevance, and a high one does not establish
causation. That caveat is saved alongside the figures so it travels with them.

In [ ]:
case_rows = {
    "TP": np.flatnonzero((y_test == 1) & (y_pred_test == 1))[:2].tolist(),
    "TN": np.flatnonzero((y_test == 0) & (y_pred_test == 0))[:1].tolist(),
    "FP": np.flatnonzero((y_test == 0) & (y_pred_test == 1))[:2].tolist(),
    "FN": np.flatnonzero((y_test == 1) & (y_pred_test == 0))[:2].tolist(),
}
case_rows = {k: v for k, v in case_rows.items() if v}

shap_summary = run_shap_analysis(
    selected_model, X_train_t, X_test_t, feature_names, cfg,
    RESULTS / "shap", case_selection=case_rows,
)
pd.DataFrame(shap_summary["global_importance"]).head(12)

In [ ]:
display(Image(filename=str(RESULTS / "shap" / "shap_summary.png")))
display(Image(filename=str(RESULTS / "shap" / "shap_importance_bar.png")))

In [ ]:
# Local explanations: why THIS patient got THIS prediction. Includes a false
# negative, which is where a local explanation is most diagnostic.
for record in shap_summary["local_explanations"]:
    print(f"--- row {record['row']} ({record['case_type']}) ---")
    for feature in record["top_features"][:4]:
        direction = "raises" if feature["shap"] > 0 else "lowers"
        print(f"    {feature['feature']:<22} = {feature['value']:>8.3f}  "
              f"{direction} risk by {abs(feature['shap']):.3f}")
    display(Image(filename=str(RESULTS / "shap" / record["figure"])))

## 16. Save Model

Three artifacts, plus metadata. They are separate on purpose: `predict.py` chains
preprocessor → model → calibrator, and keeping the calibrator distinct means
Phase 2 can access the uncalibrated score and the calibrated confidence
independently.

In [ ]:
import time

model_path = save_pickle(selected_model, MODELS / cfg.output.model_file)
preproc_path = save_pickle(preprocessor, MODELS / cfg.output.preprocessor_file)
calib_path = save_pickle(calibrator, MODELS / cfg.output.calibrator_file)

metadata = {
    "model_version": str(cfg.output.model_version),
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "modality": "clinical",
    "selected_model": selected_name,
    "selected_model_class": type(selected_model).__name__,
    "best_params": experiments[selected_name].get("best_params", {}),
    "dataset": {
        "name": str(cfg.dataset.name),
        "source": str(cfg.dataset.source),
        "target_column": str(cfg.dataset.target_column),
        "target_rule": data_report["target_rule"],
        "positive_class_meaning": str(cfg.dataset.get("positive_class_meaning", "")),
    },
    "raw_features": {
        "numeric": list(cfg.dataset.numeric_features),
        "categorical": list(cfg.dataset.categorical_features),
        "order": list(splits.X_train.columns),
    },
    "encoded_feature_names": feature_names,
    "label_mapping": {"0": "no disease", "1": "disease present"},
    "threshold": threshold,
    "threshold_info": threshold_info,
    "calibration": {"method": method, "fitted_on": "validation split",
                    "metrics": calibration_metrics},
    "split_summary": splits.summary,
    "test_metrics": test_metrics,
    "training_config": cfg.to_dict(),
    "artifacts": {"model": model_path.name, "preprocessor": preproc_path.name,
                  "calibrator": calib_path.name},
}
metadata_path = save_json(metadata, MODELS / cfg.output.metadata_file)

for path in (model_path, preproc_path, calib_path, metadata_path):
    print(f"  {path.name:<32} {path.stat().st_size / 1024:>8.1f} KB")

## 17. Save Results

In [ ]:
save_json({
    "model": selected_name,
    "test": test_metrics,
    "calibration": calibration_metrics,
    "model_comparison": comparison.to_dict(orient="records"),
    "selection": decision,
    "split": splits.summary,
}, RESULTS / "metrics.json")

# Record the run in the experiment log so the result is traceable to its config,
# git commit and hardware.
with ExperimentTracker("clinical_notebook", modality="clinical", config=cfg,
                       primary_metric="roc_auc") as run:
    run.log_params({
        "model": selected_name,
        "best_params": experiments[selected_name].get("best_params", {}),
        "threshold": threshold,
        "calibration_method": method,
        "n_train": splits.summary["sizes"]["train"],
        "n_val": splits.summary["sizes"]["val"],
        "n_test": splits.summary["sizes"]["test"],
    })
    run.log_metrics(test_metrics["at_tuned_threshold"], split="test")
    run.log_metrics(calibration_metrics["calibrated"], split="calibration")
    for name, path in metadata["artifacts"].items():
        run.log_artifact(name, MODELS / path)

print("\nfiles written:")
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(RESULTS))

In [ ]:
load_experiment_log().tail(5)[
    ["run_id", "modality", "experiment", "status", "model", "primary_metric", "primary_value"]
]

## 18. Example Inference

Loads the saved artifacts from scratch — nothing from this notebook's memory — and
runs the full chain the way a Phase 2 caller would.

The output keeps `probability` and `calibrated_confidence` separate. The first is a
ranking score; the second is a reliability statement. Fusion must weight by the
second.

In [ ]:
predictor = ClinicalPredictor.load()

example = predictor.example_patient()
print("input (a FORMAT example — not a real patient, not a dataset row):")
print(json.dumps(example, indent=2))

result = predictor.predict_one(example)
print("\noutput:")
print(json.dumps(result, indent=2))

In [ ]:
# Batch inference over the held-out test patients, straight from the saved artifacts.
batch = predictor.predict(splits.X_test.head(8))
pd.DataFrame([{k: v for k, v in r.items() if k != "notes"} for r in batch]).assign(
    actual=splits.y_test.head(8).to_numpy()
)[["prediction", "actual", "probability", "calibrated_probability",
   "calibrated_confidence", "threshold"]]

## Phase 1 clinical checklist

| Item | Where |
|---|---|
| Dataset verified | §4 · `results/clinical/dataset_report.json` |
| EDA completed | §5 · `results/clinical/eda/` |
| Leakage-free preprocessing | §7 — fit on train only |
| Stratified split | §6 · `split_summary.json`, overlap all zero |
| Logistic Regression baseline | §8 (C-A) |
| XGBoost model | §9–10 (C-B) |
| Hyperparameter tuning | §10 — RandomizedSearchCV, 5-fold stratified |
| Calibration | §14 (C-C) · `calibration_curve.png` |
| SHAP | §13 · `results/clinical/shap/` |
| Evaluation | §12 · with bootstrap CIs |
| Error analysis | §15 · `results/clinical/errors/` |
| Saved model + preprocessor + calibrator | §16 · `models/clinical/` |
| Inference script | §18 · `src/cardiosense/clinical/predict.py` |

**Next:** `02_ecg_training.ipynb` (PTB-XL, needs a GPU).

Everything here can also be run headless:

```bash
python -m cardiosense.clinical.train
```

*Research artifact. Not a medical device. Not for clinical use.*